# Public RAG Dataset Collector

This Colab notebook programmatically downloads the following public RAG evaluation datasets:

- BEIR
- Natural Questions (NQ)
- HotpotQA
- MuSiQue
- FEVER
- MultiHop-RAG
- RAGBench
- Open RAG Benchmark
- CRAG
- RGB

After **every dataset step**, the notebook prints:

- the completed dataset's on-disk size,
- the exact number of bytes, and
- remaining Colab disk space.

## Storage profiles

The default `colab_safe` profile downloads every dataset family while avoiding some exceptionally large optional components:

- a compact, diverse BEIR subset instead of all BEIR collections,
- the Natural Questions validation split instead of all splits,
- FEVER claims without the large Wikipedia-pages configuration,
- CRAG Tasks 1 and 2 without the larger Task 3 archive.

Change `PROFILE = "full"` in the configuration cell to request the larger components. The full profile can exceed the storage available in a standard Colab runtime.

> Public download availability does not imply unrestricted commercial use. Check each dataset's current license and the licenses of upstream source corpora before redistribution or commercial use.

In [ ]:
# Install download and dataset utilities.
# datasets<3 preserves compatibility with repositories that still use loading scripts.
%pip install -q "datasets>=2.19,<3" "huggingface_hub>=0.24" beir gdown tqdm pandas requests

## 1. Configuration

In [ ]:
from pathlib import Path

# "colab_safe" downloads all ten dataset families with smaller/default components.
# "full" enables the largest optional components and may exceed Colab storage.
PROFILE = "colab_safe"  #@param ["colab_safe", "full"]

ROOT = Path("/content/rag_datasets")
FORCE_REDOWNLOAD = False  #@param {type:"boolean"}

# BEIR: compact profile is deliberately diverse and avoids duplicating NQ/HotpotQA.
BEIR_COMPACT = [
    "arguana",
    "fiqa",
    "nfcorpus",
    "scifact",
    "trec-covid",
]

BEIR_ALL = [
    "arguana",
    "climate-fever",
    "cqadupstack",
    "dbpedia-entity",
    "fever",
    "fiqa",
    "hotpotqa",
    "msmarco",
    "nfcorpus",
    "nq",
    "quora",
    "robust04",
    "scidocs",
    "scifact",
    "signal1m",
    "trec-covid",
    "trec-news",
    "webis-touche2020",
]

BEIR_DATASETS = BEIR_ALL if PROFILE == "full" else BEIR_COMPACT
NQ_SPLIT = None if PROFILE == "full" else "validation"
HOTPOT_CONFIG = "distractor"
FEVER_INCLUDE_WIKI = PROFILE == "full"
CRAG_INCLUDE_TASK3 = PROFILE == "full"

ROOT.mkdir(parents=True, exist_ok=True)

print(f"Profile: {PROFILE}")
print(f"Destination: {ROOT}")
print(f"BEIR collections: {len(BEIR_DATASETS)}")
print(f"Natural Questions: {'all splits' if NQ_SPLIT is None else NQ_SPLIT}")
print(f"FEVER Wikipedia pages: {FEVER_INCLUDE_WIKI}")
print(f"CRAG Task 3: {CRAG_INCLUDE_TASK3}")

## 2. Shared download and size-reporting helpers

In [ ]:
import bz2
import shutil
import subprocess
import tarfile
import time
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display
from tqdm.auto import tqdm

DOWNLOAD_RESULTS = {}

def human_bytes(num_bytes: int) -> str:
    value = float(num_bytes)
    units = ["B", "KiB", "MiB", "GiB", "TiB"]
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:,.2f} {unit}"
        value /= 1024
    return f"{num_bytes:,} B"

def path_size_bytes(path: Path) -> int:
    """Return the physical size of regular files under path."""
    path = Path(path)
    if not path.exists():
        return 0
    if path.is_file():
        return path.stat().st_size

    total = 0
    for item in path.rglob("*"):
        try:
            if item.is_file() and not item.is_symlink():
                total += item.stat().st_size
        except FileNotFoundError:
            # A temporary file may disappear while a library finalizes a download.
            pass
    return total

def remaining_disk_bytes(path: Path = Path("/content")) -> int:
    return shutil.disk_usage(path).free

def nonempty(path: Path) -> bool:
    path = Path(path)
    return path.exists() and (path.is_file() or any(path.iterdir()))

def prepare_destination(path: Path) -> None:
    path = Path(path)
    if FORCE_REDOWNLOAD and path.exists():
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()
    path.mkdir(parents=True, exist_ok=True)

def run_command(command, cwd=None) -> None:
    printable = " ".join(map(str, command))
    print(f"$ {printable}")
    subprocess.run(
        list(map(str, command)),
        cwd=str(cwd) if cwd else None,
        check=True,
    )

def download_file(url: str, destination: Path) -> Path:
    """Stream a URL to disk with a progress bar and atomic rename."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() and destination.stat().st_size > 0 and not FORCE_REDOWNLOAD:
        print(f"Using existing file: {destination.name}")
        return destination

    temporary = destination.with_suffix(destination.suffix + ".part")
    if temporary.exists():
        temporary.unlink()

    with requests.get(url, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with temporary.open("wb") as output, tqdm(
            total=total or None,
            unit="B",
            unit_scale=True,
            desc=destination.name,
        ) as progress:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
                    progress.update(len(chunk))

    temporary.replace(destination)
    return destination

def safe_extract_tar(archive: Path, destination: Path) -> None:
    """Extract a tar archive while rejecting path traversal."""
    archive = Path(archive)
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()

    with tarfile.open(archive, "r:*") as tar:
        for member in tar.getmembers():
            target = (destination / member.name).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f"Unsafe path in archive: {member.name}")
        tar.extractall(destination)

def report_dataset(
    name: str,
    path: Path,
    started_at: float,
    status: str = "complete",
    error: str = "",
):
    size = path_size_bytes(path)
    elapsed = time.time() - started_at
    free = remaining_disk_bytes()

    DOWNLOAD_RESULTS[name] = {
        "dataset": name,
        "status": status,
        "path": str(path),
        "size_bytes": size,
        "size": human_bytes(size),
        "elapsed_seconds": round(elapsed, 1),
        "error": error,
    }

    if status == "complete":
        print(f"\n✅ {name} download step complete")
        print(f"   On-disk size: {human_bytes(size)} ({size:,} bytes)")
        print(f"   Location: {path}")
        print(f"   Remaining Colab disk: {human_bytes(free)}")
    else:
        print(f"\n❌ {name} failed: {error}")
        print(f"   Partial on-disk size: {human_bytes(size)} ({size:,} bytes)")
        print(f"   Remaining Colab disk: {human_bytes(free)}")

def execute_step(name: str, path: Path, downloader):
    print("=" * 80)
    print(f"Downloading: {name}")
    print("=" * 80)
    started = time.time()
    try:
        downloader()
        report_dataset(name, path, started, status="complete")
    except Exception as exc:
        report_dataset(name, path, started, status="failed", error=repr(exc))
        print("This notebook continues so the remaining datasets can still be collected.")

print(f"Initial free disk: {human_bytes(remaining_disk_bytes())}")

## 3. BEIR

Uses the official BEIR download utility and public UKP dataset archives.

The safe profile downloads five heterogeneous collections; the full profile requests all collections listed in the configuration.

In [ ]:
from beir import util as beir_util

beir_path = ROOT / "BEIR"

def download_beir():
    prepare_destination(beir_path)

    for dataset_name in BEIR_DATASETS:
        expected = beir_path / dataset_name
        if nonempty(expected) and not FORCE_REDOWNLOAD:
            print(f"Using existing BEIR collection: {dataset_name}")
        else:
            url = (
                "https://public.ukp.informatik.tu-darmstadt.de/"
                f"thakur/BEIR/datasets/{dataset_name}.zip"
            )
            extracted_path = Path(
                beir_util.download_and_unzip(url, str(beir_path))
            )
            print(f"Downloaded {dataset_name} -> {extracted_path}")

        collection_size = path_size_bytes(expected)
        print(
            f"   {dataset_name}: "
            f"{human_bytes(collection_size)} ({collection_size:,} bytes)"
        )

execute_step("BEIR", beir_path, download_beir)

## 4. Natural Questions (NQ)

Downloads from the official Hugging Face organization.

- `colab_safe`: validation split
- `full`: every available split

The full original Natural Questions release is exceptionally large. This step stores the processed Hugging Face dataset under the destination directory and removes its temporary conversion cache afterward.

In [ ]:
from datasets import load_dataset

nq_path = ROOT / "Natural_Questions"

def download_natural_questions():
    prepare_destination(nq_path)
    output = nq_path / "dataset"
    cache = nq_path / "_temporary_hf_cache"

    if nonempty(output) and not FORCE_REDOWNLOAD:
        print("Using existing Natural Questions dataset.")
        return

    if cache.exists():
        shutil.rmtree(cache)

    kwargs = {
        "path": "google-research-datasets/natural_questions",
        "cache_dir": str(cache),
    }
    if NQ_SPLIT is not None:
        kwargs["split"] = NQ_SPLIT

    dataset = load_dataset(**kwargs)
    dataset.save_to_disk(str(output))

    del dataset
    if cache.exists():
        shutil.rmtree(cache)

execute_step("Natural Questions (NQ)", nq_path, download_natural_questions)

## 5. HotpotQA

Downloads the `distractor` configuration from the official Hugging Face dataset repository. It includes supporting-fact annotations and is practical for retrieval-plus-reasoning evaluation.

In [ ]:
hotpot_path = ROOT / "HotpotQA"

def download_hotpotqa():
    prepare_destination(hotpot_path)
    output = hotpot_path / "dataset"
    cache = hotpot_path / "_temporary_hf_cache"

    if nonempty(output) and not FORCE_REDOWNLOAD:
        print("Using existing HotpotQA dataset.")
        return

    if cache.exists():
        shutil.rmtree(cache)

    dataset = load_dataset(
        "hotpotqa/hotpot_qa",
        HOTPOT_CONFIG,
        cache_dir=str(cache),
    )
    dataset.save_to_disk(str(output))

    del dataset
    if cache.exists():
        shutil.rmtree(cache)

execute_step("HotpotQA", hotpot_path, download_hotpotqa)

## 6. MuSiQue

Clones the official repository and runs its official `download_data.sh` script. Downloaded files remain under the repository's `data/` directory.

In [ ]:
musique_path = ROOT / "MuSiQue"

def download_musique():
    if FORCE_REDOWNLOAD and musique_path.exists():
        shutil.rmtree(musique_path)

    if not (musique_path / ".git").exists():
        run_command([
            "git", "clone", "--depth", "1",
            "https://github.com/StonyBrookNLP/musique.git",
            musique_path,
        ])

    data_path = musique_path / "data"
    if nonempty(data_path) and not FORCE_REDOWNLOAD:
        print("Using existing MuSiQue data directory.")
        return

    run_command(["bash", "download_data.sh"], cwd=musique_path)

execute_step("MuSiQue", musique_path, download_musique)

## 7. FEVER

Downloads FEVER through its Hugging Face loader.

- Both profiles download the `v1.0` claims/evidence configuration.
- The `full` profile additionally downloads `wiki_pages`, which requires substantially more disk space.

In [ ]:
fever_path = ROOT / "FEVER"

def save_hf_configuration(repo_id: str, config: str, output: Path):
    cache = output.parent / f"_cache_{config.replace('.', '_')}"
    if nonempty(output) and not FORCE_REDOWNLOAD:
        print(f"Using existing configuration: {config}")
        return

    if cache.exists():
        shutil.rmtree(cache)

    dataset = load_dataset(
        repo_id,
        config,
        cache_dir=str(cache),
        trust_remote_code=True,
    )
    dataset.save_to_disk(str(output))
    del dataset

    if cache.exists():
        shutil.rmtree(cache)

def download_fever():
    prepare_destination(fever_path)

    save_hf_configuration(
        "fever/fever",
        "v1.0",
        fever_path / "v1_0",
    )

    if FEVER_INCLUDE_WIKI:
        save_hf_configuration(
            "fever/fever",
            "wiki_pages",
            fever_path / "wiki_pages",
        )
    else:
        print("Skipping FEVER wiki_pages in colab_safe profile.")

execute_step("FEVER", fever_path, download_fever)

## 8. MultiHop-RAG

Downloads a local snapshot of the public Hugging Face dataset repository.

In [ ]:
from huggingface_hub import snapshot_download

multihop_rag_path = ROOT / "MultiHop-RAG"

def download_multihop_rag():
    if FORCE_REDOWNLOAD and multihop_rag_path.exists():
        shutil.rmtree(multihop_rag_path)

    snapshot_download(
        repo_id="yixuantt/MultiHopRAG",
        repo_type="dataset",
        local_dir=str(multihop_rag_path),
        local_dir_use_symlinks=False,
    )

execute_step("MultiHop-RAG", multihop_rag_path, download_multihop_rag)

## 9. RAGBench

Downloads the complete public Hugging Face repository snapshot, including its available domain configurations.

In [ ]:
ragbench_path = ROOT / "RAGBench"

def download_ragbench():
    if FORCE_REDOWNLOAD and ragbench_path.exists():
        shutil.rmtree(ragbench_path)

    snapshot_download(
        repo_id="galileo-ai/ragbench",
        repo_type="dataset",
        local_dir=str(ragbench_path),
        local_dir_use_symlinks=False,
    )

execute_step("RAGBench", ragbench_path, download_ragbench)

## 10. Open RAG Benchmark

Downloads the raw public repository snapshot. A snapshot download is used because the benchmark contains source-oriented files useful for PDF and multimodal ingestion tests.

In [ ]:
open_rag_path = ROOT / "Open_RAG_Benchmark"

def download_open_rag_benchmark():
    if FORCE_REDOWNLOAD and open_rag_path.exists():
        shutil.rmtree(open_rag_path)

    snapshot_download(
        repo_id="vectara/open_ragbench",
        repo_type="dataset",
        local_dir=str(open_rag_path),
        local_dir_use_symlinks=False,
    )

execute_step("Open RAG Benchmark", open_rag_path, download_open_rag_benchmark)

## 11. CRAG

Downloads official development data directly from the CRAG repository.

- Both profiles download and decompress Tasks 1 and 2.
- The `full` profile also downloads the four Task 3 archive parts, joins them, and safely extracts the resulting archive.

In [ ]:
crag_path = ROOT / "CRAG"
crag_base_url = (
    "https://github.com/facebookresearch/CRAG/raw/"
    "refs/heads/main/data"
)

def download_crag():
    prepare_destination(crag_path)

    task12_output = crag_path / "crag_task_1_and_2_dev_v4.jsonl"
    task12_archive = crag_path / "crag_task_1_and_2_dev_v4.jsonl.bz2"

    if not nonempty(task12_output) or FORCE_REDOWNLOAD:
        download_file(
            f"{crag_base_url}/crag_task_1_and_2_dev_v4.jsonl.bz2",
            task12_archive,
        )
        print("Decompressing CRAG Tasks 1 and 2...")
        with bz2.open(task12_archive, "rb") as source, task12_output.open("wb") as target:
            shutil.copyfileobj(source, target)
        task12_archive.unlink(missing_ok=True)
    else:
        print("Using existing CRAG Tasks 1 and 2 data.")

    if not CRAG_INCLUDE_TASK3:
        print("Skipping CRAG Task 3 in colab_safe profile.")
        return

    task3_directory = crag_path / "task3"
    if nonempty(task3_directory) and not FORCE_REDOWNLOAD:
        print("Using existing CRAG Task 3 data.")
        return

    part_paths = []
    for part_number in range(1, 5):
        part_name = f"crag_task_3_dev_v4.tar.bz2.part{part_number}"
        part_path = crag_path / part_name
        download_file(f"{crag_base_url}/{part_name}", part_path)
        part_paths.append(part_path)

    joined_archive = crag_path / "crag_task_3_dev_v4.tar.bz2"
    print("Joining CRAG Task 3 archive parts...")
    with joined_archive.open("wb") as output:
        for part_path in part_paths:
            with part_path.open("rb") as part:
                shutil.copyfileobj(part, output)

    print("Extracting CRAG Task 3...")
    safe_extract_tar(joined_archive, task3_directory)

    for part_path in part_paths:
        part_path.unlink(missing_ok=True)
    joined_archive.unlink(missing_ok=True)

execute_step("CRAG", crag_path, download_crag)

## 12. RGB

Clones the official repository, whose `data/` directory contains the English and Chinese robustness benchmark files.

In [ ]:
rgb_path = ROOT / "RGB"

def download_rgb():
    if FORCE_REDOWNLOAD and rgb_path.exists():
        shutil.rmtree(rgb_path)

    if not (rgb_path / ".git").exists():
        run_command([
            "git", "clone", "--depth", "1", "--branch", "master",
            "https://github.com/chen700564/RGB.git",
            rgb_path,
        ])
    else:
        print("Using existing RGB repository.")

execute_step("RGB", rgb_path, download_rgb)

## 13. Download summary and machine-readable manifest

In [ ]:
summary = pd.DataFrame(DOWNLOAD_RESULTS.values())

if summary.empty:
    print("No download steps have been run in this session.")
else:
    ordered_columns = [
        "dataset",
        "status",
        "size",
        "size_bytes",
        "elapsed_seconds",
        "path",
        "error",
    ]
    summary = summary[ordered_columns].sort_values("dataset").reset_index(drop=True)
    display(summary)

    successful = summary[summary["status"] == "complete"]
    total_bytes = int(successful["size_bytes"].sum())

    manifest_path = ROOT / "download_manifest.csv"
    summary.to_csv(manifest_path, index=False)

    print(
        "Total completed dataset footprint: "
        f"{human_bytes(total_bytes)} ({total_bytes:,} bytes)"
    )
    print(f"Manifest written to: {manifest_path}")
    print(f"Remaining Colab disk: {human_bytes(remaining_disk_bytes())}")

## Optional: archive the collected datasets

The following cell creates one compressed archive. It is disabled by default because archive creation temporarily requires additional disk space.

In [ ]:
CREATE_ARCHIVE = False  #@param {type:"boolean"}

if CREATE_ARCHIVE:
    archive_base = Path("/content/rag_datasets")
    archive_path = shutil.make_archive(
        str(archive_base),
        "gztar",
        root_dir=str(ROOT.parent),
        base_dir=ROOT.name,
    )
    archive_size = Path(archive_path).stat().st_size
    print(f"Archive: {archive_path}")
    print(f"Archive size: {human_bytes(archive_size)} ({archive_size:,} bytes)")
else:
    print("Archive creation is disabled.")